# Play with detection

In [ ]:
import numpy as np
from theia.detection.active import get_rad_pd
from theia.types import Point, Polarization, Radar, Target


transmitter = Radar(
    id=585,
    point=Point(
        lat=47.36700085728634,
        lon=8.537724304199216,
        alt=407.83600886023686,
    ),
    power=20000,
    erp=800,
    antenna_height=10.0,
    diameter=2.0,
    frequency=1000.0,
    pulse_width=1,
    cpi_pulses=1,
    bandwidth=1,
    pfa=1e-6,
    min_elevation=-20.0,
    max_elevation=60.0,
    rotation_time=10.0,
    polarization=Polarization.HORIZONTAL,
)

target_flight_height: float = 1000.0
target_cross_section: float = 1.0

target = Target(
    id=1,
    point=Point(
        lat=47.348,
        lon=8.6266,
        alt=1000.0,
    ),
    cross_section=target_cross_section,
    vlon=250.0,
    vlat=0.0,
    vz=0.0,
)

target2 = Target(
    id=2,
    point=Point(
        lat=47.406,
        lon=8.3926,
        alt=1000.0,
    ),
    cross_section=target_cross_section,
    vlon=250.0,
    vlat=0.0,
    vz=0.0,
)

In [ ]:
%%timeit
get_rad_pd(transmitter, target)

# Play with visualisation

In [ ]:
from theia.coverage import calculate_coverage
from theia.radar_equation import radar_eq_max_dist


coverage = calculate_coverage(
    transmitter.point,
    radar_eq_max_dist(transmitter, target_cross_section),
    1000.0,
    d_theta=1,
    dist_res=300,
)

In [ ]:
import folium
from folium.plugins import MousePosition


map = folium.Map(location=(transmitter.lat, transmitter.lon))

folium.Marker(
    (transmitter.lat, transmitter.lon),
    color="blue",
).add_to(map)
folium.Marker(
    (target.lat, target.lon),
    icon=folium.Icon(color="red"),
).add_to(map)
folium.Marker(
    (target2.lat, target2.lon),
    icon=folium.Icon(color="green"),
).add_to(map)
folium.GeoJson(coverage).add_to(map)

formatter = "function(num) {return L.Util.formatNum(num, 3) + ' º ';};"

MousePosition(
    position="topright",
    separator=" | ",
    empty_string="NaN",
    lng_first=True,
    num_digits=20,
    prefix="Coordinates:",
    lat_formatter=formatter,
    lng_formatter=formatter,
).add_to(map)

map.save("map.html")

In [ ]:
from scipy.interpolate import CubicSpline
import numpy as np

target_height = 1000.0

checkpoints = [
    Point(lat=47.2196, lon=8.4128, alt=target_height),
    Point(lat=47.3090, lon=8.6407, alt=target_height),
    Point(lat=47.4810, lon=8.6572, alt=target_height),
]

f = CubicSpline(
    np.linspace(0, 1, len(checkpoints)), [(p.lat, p.lon) for p in checkpoints]
)

line = f(np.linspace(0, 1, 100))

In [ ]:
import folium
from folium.plugins import MousePosition

formatter = "function(num) {return L.Util.formatNum(num, 3) + ' º ';};"

map = folium.Map(
    location=(transmitter.lat, transmitter.lon),
    # width=6*160,
    # height=6*90,
)
folium.GeoJson(coverage).add_to(map)
folium.CircleMarker(location=(transmitter.lat, transmitter.lon), color="blue", fill=True).add_to(
    map
)
folium.LatLngPopup().add_to(map)

for p in checkpoints:
    folium.CircleMarker(location=(p.lat, p.lon), color="red", fill=True).add_to(map)
folium.PolyLine(line, color="red", tooltip="Path").add_to(map)

map.save("map.html")